# Predictive maintenance - medallion pipeline

End-to-end exploration organised in **3 chapters** — **1. BRONZE** (raw data) > **2. SILVER** (treated data) > **3. GOLD** (ready for training). Each source follows the same template: **PREVIEW** (per-feature understanding) > **PROCESSING** (per-feature transformation) > **OVERVIEW** (permanent + inline analysis); layer-level / cross-source material sits in a per-chapter appendix. Reproducible run: `uv run python scripts/run_pipeline.py`.

## 0. Setup

In [ ]:
import sys
import tempfile
from pathlib import Path

%load_ext autoreload
%autoreload 2

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

%matplotlib inline

from src import config
from src.framework.common.env import load_dotenv
# Rendering helpers + shared state (SPECS, bronze, silver, gold, bronze_flagged, silver_stats):
from src.usecase.notebook.render import *  # noqa: F403

load_dotenv(PROJECT_ROOT / '.env')

## 1. BRONZE (raw data)

### 1.1 Incidents

#### 1.1.1 PREVIEW

In [ ]:
spec = SPECS['incidents']
bronze['incidents'] = spec.load_bronze()
print('incidents bronze', bronze['incidents'].shape)
show_per_feature_spec(spec, bronze['incidents'], spec.bronze_numeric)

#### 1.1.2 PROCESSING

In [ ]:
show_bronze_processing('incidents')

#### 1.1.3 OVERVIEW

##### Permanent Analysis

In [ ]:
show_overview(SPECS['incidents'], bronze['incidents'])

##### Inline Analysis

Incident volume over time, one curve per machine (weekly), then for MACH-03 and MACH-13.

In [ ]:
# Inline (notebook-only): number of incidents over time, one curve per machine.
_idf = bronze['incidents'].copy()
_idf['date'] = pd.to_datetime(_idf['date'], errors='coerce')
_counts = (
    _idf.dropna(subset=['date'])
    .groupby([pd.Grouper(key='date', freq='W'), 'machine_id'])
    .size()
    .unstack('machine_id')
    .fillna(0)
)
fig, ax = plt.subplots(figsize=(13, 6))
for mc in sorted(_counts.columns):
    ax.plot(_counts.index, _counts[mc], linewidth=0.9, label=str(mc))
ax.set_title('Incidents over time by machine (weekly count)')
ax.set_xlabel('date')
ax.set_ylabel('number of incidents')
ax.grid(True, alpha=0.3)
ax.legend(title='machine_id', fontsize=6, ncol=2, loc='upper left', bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()

In [ ]:
# Inline (notebook-only): weekly incident count for two specific machines.
_idf = bronze['incidents'].copy()
_idf['date'] = pd.to_datetime(_idf['date'], errors='coerce')
_counts = (
    _idf.dropna(subset=['date'])
    .groupby([pd.Grouper(key='date', freq='W'), 'machine_id'])
    .size()
    .unstack('machine_id')
    .fillna(0)
)
for _mc in ['MACH-03', 'MACH-13']:
    if _mc not in _counts.columns:
        print(f'{_mc}: no data')
        continue
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(_counts.index, _counts[_mc], color='#4C72B0', linewidth=1.2)
    ax.set_title(f'Incidents over time - {_mc} (weekly count)')
    ax.set_xlabel('date')
    ax.set_ylabel('number of incidents')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### 1.2 Telemetry

#### 1.2.1 PREVIEW

In [ ]:
spec = SPECS['telemetry']
bronze['telemetry'] = spec.load_bronze()
print('telemetry bronze', bronze['telemetry'].shape)
show_per_feature_spec(spec, bronze['telemetry'], spec.bronze_numeric)

#### 1.2.2 PROCESSING

In [ ]:
show_bronze_processing('telemetry')

#### 1.2.3 OVERVIEW

##### Permanent Analysis

In [ ]:
show_overview(SPECS['telemetry'], bronze['telemetry'])

##### Inline Analysis

One-week zoom on the weekly cyclicity (4 measures + production): mean across machines, by machine, then for MACH-03 and MACH-13.

In [ ]:
# Inline (notebook-only): one-week zoom to inspect the weekly cyclicity.
_tdf = bronze['telemetry'].copy()
_tdf['t'] = pd.to_datetime(_tdf['timestamp'], errors='coerce')
_start = _tdf['t'].min().normalize()
_week = _tdf[(_tdf['t'] >= _start) & (_tdf['t'] < _start + pd.Timedelta(days=7))]
_measures = ['temperature_c', 'pressure_bar', 'voltage_mean_v', 'rotation_mean_rpm']

# 1) the 4 physical measures over the week (hourly mean across machines)
_hourly = _week.set_index('t')[_measures].resample('h').mean()
fig, axes = plt.subplots(len(_measures), 1, figsize=(12, 8), sharex=True)
for ax, m in zip(axes, _measures):
    ax.plot(_hourly.index, _hourly[m], color='#4C72B0', linewidth=1)
    ax.set_ylabel(m)
    ax.grid(True, alpha=0.3)
axes[0].set_title(f'Telemetry measures - 1 week zoom ({_start.date()}, hourly mean, all machines)')
axes[-1].set_xlabel('timestamp')
plt.tight_layout()
plt.show()

# 2) piece production over the same week (hourly mean across machines)
_prod = _week.set_index('t')['pieces_produced'].resample('h').mean()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(_prod.index, _prod, color='#C44E52', linewidth=1)
ax.set_title(f'Piece production - 1 week zoom ({_start.date()}, hourly mean, all machines)')
ax.set_xlabel('timestamp')
ax.set_ylabel('pieces_produced')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Inline (notebook-only): same one-week zoom, but one curve per machine.
_tdf = bronze['telemetry'].copy()
_tdf['t'] = pd.to_datetime(_tdf['timestamp'], errors='coerce')
_start = _tdf['t'].min().normalize()
_week = _tdf[(_tdf['t'] >= _start) & (_tdf['t'] < _start + pd.Timedelta(days=7))]
_measures = ['temperature_c', 'pressure_bar', 'voltage_mean_v', 'rotation_mean_rpm']
_machines = sorted(_week['machine_id'].dropna().unique())

# 1) the 4 measures over the week, one line per machine
fig, axes = plt.subplots(len(_measures), 1, figsize=(13, 9), sharex=True)
for ax, m in zip(axes, _measures):
    piv = _week.pivot_table(index='t', columns='machine_id', values=m, aggfunc='mean')
    for mc in _machines:
        if mc in piv.columns:
            ax.plot(piv.index, piv[mc], linewidth=0.7, label=str(mc))
    ax.set_ylabel(m)
    ax.grid(True, alpha=0.3)
axes[0].set_title(f'Telemetry measures - 1 week zoom, by machine ({_start.date()})')
axes[-1].set_xlabel('timestamp')
axes[0].legend(title='machine_id', fontsize=6, ncol=2, loc='upper left', bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()

# 2) piece production over the same week, one line per machine
piv = _week.pivot_table(index='t', columns='machine_id', values='pieces_produced', aggfunc='mean')
fig, ax = plt.subplots(figsize=(13, 5))
for mc in _machines:
    if mc in piv.columns:
        ax.plot(piv.index, piv[mc], linewidth=0.7, label=str(mc))
ax.set_title(f'Piece production - 1 week zoom, by machine ({_start.date()})')
ax.set_xlabel('timestamp')
ax.set_ylabel('pieces_produced')
ax.grid(True, alpha=0.3)
ax.legend(title='machine_id', fontsize=6, ncol=2, loc='upper left', bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()

In [ ]:
# Inline (notebook-only): "measures - 1 week zoom" for two specific machines.
_tdf = bronze['telemetry'].copy()
_tdf['t'] = pd.to_datetime(_tdf['timestamp'], errors='coerce')
_start = _tdf['t'].min().normalize()
_week = _tdf[(_tdf['t'] >= _start) & (_tdf['t'] < _start + pd.Timedelta(days=7))]
_measures = ['temperature_c', 'pressure_bar', 'voltage_mean_v', 'rotation_mean_rpm']
for _mc in ['MACH-03', 'MACH-13']:
    _w = _week[_week['machine_id'] == _mc]
    if _w.empty:
        print(f'{_mc}: no data in window')
        continue
    _hourly = _w.set_index('t')[_measures].resample('h').mean()
    fig, axes = plt.subplots(len(_measures), 1, figsize=(12, 8), sharex=True)
    for ax, m in zip(axes, _measures):
        ax.plot(_hourly.index, _hourly[m], color='#4C72B0', linewidth=1)
        ax.set_ylabel(m)
        ax.grid(True, alpha=0.3)
    axes[0].set_title(f'Telemetry measures - 1 week zoom - {_mc} ({_start.date()})')
    axes[-1].set_xlabel('timestamp')
    plt.tight_layout()
    plt.show()

### 1.3 Machines/machine

#### 1.3.1 PREVIEW

**Browse the `machine` table** (notebook-only): run the cell, then open the **Variables**
view and click the grid icon next to `machine` to sort / filter every row.

In [ ]:
# Inline (notebook-only): browse the raw `machine` table (Variables pane -> Data Viewer).
from src.usecase.sources.machines.loader import build_engine, load_machine_referential

pd.set_option('display.max_columns', None)
machine = load_machine_referential(build_engine())   # referential: one row per machine (15)
print('machine', machine.shape)
display(machine)

In [ ]:
spec = SPECS['machine']
bronze['machine'] = spec.load_bronze()
print('machine bronze', bronze['machine'].shape)
show_per_feature_spec(spec, bronze['machine'], spec.bronze_numeric)

#### 1.3.2 PROCESSING

In [ ]:
show_bronze_processing('machine')

#### 1.3.3 OVERVIEW

##### Permanent Analysis

In [ ]:
from src.usecase.sources.machines.overview import machine_tree_markdown

# Overview as a tree: location -> production_line -> criticality -> model -> machine_id
display(Markdown(machine_tree_markdown(bronze['machine'])))

##### Inline Analysis

_(inline analysis to come)_

### 1.4 Machines/maintenance

#### 1.4.1 PREVIEW

**Browse the `maintenance` table** (notebook-only): run the cell, then open the
**Variables** view and click the grid icon next to `maintenance` to sort / filter rows.

In [ ]:
# Inline (notebook-only): browse the raw `maintenance` table (Variables pane -> Data Viewer).
from src.usecase.sources.machines.loader import build_engine, load_maintenance

pd.set_option('display.max_columns', None)
maintenance = load_maintenance(build_engine())       # maintenance events (1562)
print('maintenance', maintenance.shape)
with pd.option_context('display.max_rows', None):
    display(maintenance)

In [ ]:
spec = SPECS['machines']
bronze['machines'] = spec.load_bronze()
print('machines bronze', bronze['machines'].shape)
show_per_feature_spec(spec, bronze['machines'], spec.bronze_numeric)

#### 1.4.2 PROCESSING

In [ ]:
show_bronze_processing('machines')

#### 1.4.3 OVERVIEW

##### Permanent Analysis

In [ ]:
show_overview(SPECS['machines'], bronze['machines'])

##### Inline Analysis

_(inline analysis to come)_

### 1.5 Layer & cross-source (appendix)

Layer-level steps that span sources: database upload, cross-source coherence, cross-source analysis, and the Bronze synthesis.

#### 1.5.1 Ingestion statistics

Global recap across the 4 Bronze sources: rows ingested from the DataLake, valid (`parse_ok`) and rejected (`parse_ok=False`) counts. Requires the PROCESSING cells above to have run.

In [ ]:
from src.framework.ingestion.stats import global_summary_markdown

# Global ingestion recap (per source: original=ingested, valid, rejected).
display(Markdown(global_summary_markdown(bronze_flagged)))

#### 1.5.2 Cross-source coherence

In [ ]:
from src.framework.common.coherence import cross_checks_markdown

# Cross-source coherence checks (each shown as ##### <name> (OK/NOK) + explanations).
display(Markdown(cross_checks_markdown(bronze)))

In [ ]:
from src.framework.common.coherence import plot_pieces_vs_capacity

# Illustrate "pieces_produced within machine capacity": one panel per machine, real cumulative
# production (telemetry) vs the theoretical maximum from the daily capacity (machines/machine).
_out = Path(tempfile.mkdtemp(prefix='nb_capacity_'))
for _png in plot_pieces_vs_capacity(bronze, _out):
    display(Image(filename=str(_png)))

#### 1.5.3 Cross-source analysis

In [ ]:
# Cross-source analysis from the database (requires 1.5 Bronze - Database upload).
from src.framework.db.engine import get_engine, is_available

if not is_available():
    print('PostgreSQL unavailable - run 1.5 Bronze - Database upload first (docker compose up -d).')
else:
    _eng = get_engine()
    _have = set(pd.read_sql(
        "SELECT table_name FROM information_schema.tables WHERE table_schema = 'bronze'", _eng
    )['table_name'])
    if not {'incidents', 'telemetry', 'machine', 'maintenance'} <= _have:
        print('Bronze tables missing - run 1.5 Bronze - Database upload first.')
    else:
        _inc = pd.read_sql('SELECT machine_id, COUNT(*) AS n_incidents FROM bronze.incidents GROUP BY machine_id', _eng)
        _mnt = pd.read_sql('SELECT machine_id, COUNT(*) AS n_maintenance FROM bronze.maintenance GROUP BY machine_id', _eng)
        _tel = pd.read_sql('SELECT machine_id, AVG(temperature_c) AS mean_temperature_c FROM bronze.telemetry GROUP BY machine_id', _eng)
        _mac = pd.read_sql('SELECT machine_id, criticality FROM bronze.machine', _eng)
        _prof = (_mac.merge(_inc, on='machine_id', how='left')
                     .merge(_mnt, on='machine_id', how='left')
                     .merge(_tel, on='machine_id', how='left'))
        _prof[['n_incidents', 'n_maintenance']] = _prof[['n_incidents', 'n_maintenance']].fillna(0)
        _colors = {'LOW': '#55A868', 'MEDIUM': '#DD8452', 'HIGH': '#C44E52'}

        def _scatter(xc, yc, xl, yl, title):
            display(Markdown(f'##### {title}'))
            fig, ax = plt.subplots(figsize=(9, 6))
            for crit, g in _prof.groupby('criticality'):
                ax.scatter(g[xc], g[yc], s=90, c=_colors.get(crit, '#888'),
                           label=str(crit), edgecolor='black', zorder=3)
            for _, r in _prof.iterrows():
                ax.annotate(str(r['machine_id']).replace('MACH-', ''), (r[xc], r[yc]),
                            fontsize=7, xytext=(3, 3), textcoords='offset points')
            ax.set_xlabel(xl)
            ax.set_ylabel(yl)
            ax.set_title(title)
            ax.grid(True, alpha=0.3)
            ax.legend(title='criticality', fontsize=7)
            plt.tight_layout()
            plt.show()

        _scatter('n_incidents', 'n_maintenance', 'number of incidents', 'number of maintenances',
                 'Incidents vs maintenances per machine')
        _scatter('n_incidents', 'mean_temperature_c', 'number of incidents', 'mean temperature (deg C)',
                 'Mean temperature vs incidents per machine')

In [ ]:
# MACH-03 over one month: telemetry measures + incidents + reactive maintenances overlaid.
from matplotlib.lines import Line2D

_machine = 'MACH-03'
_measures = ['temperature_c', 'pressure_bar', 'voltage_mean_v', 'rotation_mean_rpm']
display(Markdown(
    f'##### {_machine} - telemetry measures with incidents and reactive maintenances (1 month)'
))

# Reactive maintenances for this machine (defines the 1-month window) - maintenance_at is UTC.
_mr = bronze['machines'].query("machine_id == @_machine and maintenance_type == 'reactive'").copy()
_mr['t'] = pd.to_datetime(_mr['maintenance_at'], errors='coerce').dt.tz_localize(None)
_mr = _mr.dropna(subset=['t']).sort_values('t')
_start = _mr['t'].iloc[0].normalize().replace(day=1)        # first day of the first reactive month
_end = _start + pd.DateOffset(months=1)

# Telemetry (hourly measures), incidents (event dates), reactive maintenances - within the window.
_tw = bronze['telemetry'].query("machine_id == @_machine").copy()
_tw['t'] = pd.to_datetime(_tw['timestamp'], errors='coerce')
_tw = _tw[(_tw['t'] >= _start) & (_tw['t'] < _end)].sort_values('t')

_iw = bronze['incidents'].query("machine_id == @_machine").copy()
_iw['t'] = pd.to_datetime(_iw['date'], errors='coerce')
_iw = _iw[(_iw['t'] >= _start) & (_iw['t'] < _end)]

_mw = _mr[(_mr['t'] >= _start) & (_mr['t'] < _end)]

fig, axes = plt.subplots(len(_measures), 1, figsize=(13, 10), sharex=True)
for ax, m in zip(axes, _measures):
    ax.plot(_tw['t'], _tw[m], color='#4C72B0', linewidth=1)
    for x in _iw['t']:
        ax.axvline(x, color='#C44E52', linestyle='--', alpha=0.6, linewidth=1)
    for x in _mw['t']:
        ax.axvline(x, color='black', linestyle=':', alpha=0.8, linewidth=1.2)
    ax.set_ylabel(m)
    ax.grid(True, alpha=0.3)
axes[0].set_title(f'{_machine} - 1 month ({_start.date()}): telemetry measures + incidents + reactive maintenances')
axes[0].legend(handles=[
    Line2D([0], [0], color='#4C72B0', label='telemetry measure'),
    Line2D([0], [0], color='#C44E52', ls='--', label='incident'),
    Line2D([0], [0], color='black', ls=':', label='reactive maintenance'),
], loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=8)
axes[-1].set_xlabel('time')
print(f'{_machine} {_start.date()}..{_end.date()} | telemetry {len(_tw)} | incidents {len(_iw)} | reactive {len(_mw)}')
plt.tight_layout()
plt.show()

#### 1.5.4 Bronze synthesis

In [ ]:
from src.framework.common.synthesis import bronze_synthesis_markdown

# Recap table: every feature per source, status + suggested Bronze -> Silver processing.
display(Markdown(bronze_synthesis_markdown(bronze)))

## 2. SILVER (treated data)

### 2.1 Incidents

#### 2.1.1 PREVIEW

In [ ]:
show_silver_preview('incidents')

#### 2.1.2 PROCESSING

In [ ]:
show_silver_processing('incidents')

#### 2.1.3 OVERVIEW

##### Permanent Analysis

In [ ]:
show_silver_overview('incidents')

##### Inline Analysis

_(inline analysis to come)_

### 2.2 Telemetry

#### 2.2.1 PREVIEW

In [ ]:
show_silver_preview('telemetry')

#### 2.2.2 PROCESSING

In [ ]:
show_silver_processing('telemetry')

#### 2.2.3 OVERVIEW

##### Permanent Analysis

In [ ]:
show_silver_overview('telemetry')

##### Inline Analysis

_(inline analysis to come)_

### 2.3 Maintenance

#### 2.3.1 PREVIEW

In [ ]:
show_silver_preview('machines')

#### 2.3.2 PROCESSING

In [ ]:
show_silver_processing('machines')

#### 2.3.3 OVERVIEW

##### Permanent Analysis

In [ ]:
show_silver_overview('machines')

##### Inline Analysis

_(inline analysis to come)_

### 2.4 Layer overview (appendix)

Layer-wide overview of the treated Silver data.

In [ ]:
from src.usecase.silver.refine import silver_summary_markdown
display(Markdown(silver_summary_markdown(silver_stats)))
show_global(silver, 'silver')

## 3. GOLD (ready for training)

Single training-ready table `gold.features`, grain **(machine_id, 1-hour window)**. Decision instant **t = window_end**; a **failure = incident of severity >= 4**.

### 3.1 Features

#### 3.1.1 PREVIEW

In [ ]:
show_gold_usecase()

#### 3.1.2 PROCESSING

In [ ]:
show_gold_build()

**Feature groups (what / why):**

- **Identifiers (4)** — `machine_id`, `window_start`, `window_end`, `split_set` (=`"train"`).
- **Memory (105)** — 5 telemetry measures x {2,3,4,6,12,24,48}h x {mean, max, std}: `<measure>_{mean|max|std}_<H>h`. *Why:* recent level & dispersion of each signal.
- **Trend (25)** — 5 measures x {2,3,4,5,6}h: `<measure>_trend_<H>h` (OLS slope per hour). *Why:* short-term drift/direction.
- **Anomaly (10)** — per measure: `<measure>_z_24h` (z-score vs trailing 24h) and `<measure>_z_machine` (z-score vs the machine over all data). *Why:* how unusual the current value is.
- **Context — incidents (11)** — {6,12,24,48h,7d}: `inc_count_<H>`, `inc_sevmax_<H>`; + `inc_hours_since_last`. *Why:* recent incident pressure & recency.
- **Context — signals (45)** — 9 `type_*` signals x {6,12,24,48h,7d}: `sig_<name>_count_<H>`. *Why:* which anomaly types fired recently.
- **Context — maintenance (12)** — {5,10,20,30,60d}: `mnt_corr_count_<D>`, `mnt_prov_count_<D>`; + `mnt_corr_days_since_last`, `mnt_prov_days_since_last` (corrective = reactive). *Why:* maintenance history.
- **Labels (4)** — `label_failure_next_{6,12,24,48}h` (1 if a failure occurs in `(t, t+H]`, **NaN** when the future window is censored at the series end). *Why:* prediction targets.

_z_machine uses full-history stats (mild leakage) -> recompute on the train split later. Edges: std/trend with < 2 points -> NaN; counts -> 0; recency -> NaN before the first event._

_Placeholder — extend the builder `src/gold/features.py` with further preparation (target definition, extra aggregations, feature selection…). To be completed._

#### 3.1.3 OVERVIEW

##### Permanent Analysis

In [ ]:
show_gold_overview()

##### Inline Analysis

Notebook-only: for each machine, 10 random incidents (fixed seed -> reproducible), each shown as the 4 telemetry measures over a 7-day window (+/-3.5 days). The centre incident is dashed; any other incident in the window is dotted.

In [ ]:
# Inline (notebook-only): telemetry over a window centred on each of N random incidents.
from matplotlib.lines import Line2D

_MEASURES = ['temperature_c', 'pressure_bar', 'voltage_mean_v', 'rotation_mean_rpm']


def _window_label(half_hours):
    """Human-readable span for a +/-half_hours window (e.g. 12 -> '24h', 84 -> '7d')."""
    total = 2 * half_hours
    return f'{total // 24}d' if total >= 48 and total % 24 == 0 else f'{total}h'


def plot_incident_window(machine, event, inc_df, tel_df, half_hours=12):
    """One figure (4 measure subplots) for the +/-half_hours window around `event`."""
    span = _window_label(half_hours)
    start, end = event - pd.Timedelta(hours=half_hours), event + pd.Timedelta(hours=half_hours)
    tw = tel_df[tel_df['machine_id'] == machine].copy()
    tw['t'] = pd.to_datetime(tw['timestamp'], errors='coerce')
    tw = tw[(tw['t'] >= start) & (tw['t'] <= end)].sort_values('t')
    others = inc_df[(inc_df['t'] >= start) & (inc_df['t'] <= end) & (inc_df['t'] != event)]['t']
    fig, axes = plt.subplots(len(_MEASURES), 1, figsize=(11, 7), sharex=True)
    for ax, m in zip(axes, _MEASURES):
        ax.plot(tw['t'], tw[m], color='#4C72B0', marker='o', markersize=2.5, linewidth=1)
        ax.axvline(event, color='#C44E52', linestyle='--', linewidth=1.2)
        for x in others:
            ax.axvline(x, color='#C44E52', linestyle=':', alpha=0.4, linewidth=1)
        ax.set_ylabel(m, fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelsize=7)
    axes[0].set_title(f'{machine} - telemetry over {span} around incident {event:%Y-%m-%d %H:%M}', fontsize=9)
    axes[0].legend(handles=[
        Line2D([0], [0], color='#4C72B0', marker='o', label='telemetry measure'),
        Line2D([0], [0], color='#C44E52', ls='--', label='incident (centre)'),
        Line2D([0], [0], color='#C44E52', ls=':', alpha=0.4, label='other incident in window'),
    ], loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=7)
    axes[-1].set_xlabel('timestamp')
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def show_incident_windows(machine, n=10, seed=0, half_hours=12):
    """Plot `n` random incident windows (reproducible via `seed`) spread over the full range."""
    inc = silver['incidents'].query('machine_id == @machine').copy()
    inc['t'] = pd.to_datetime(inc['date'].astype(str) + ' ' + inc['time'].astype(str), errors='coerce')
    inc = inc.dropna(subset=['t'])
    sample = inc.sample(n=min(n, len(inc)), random_state=seed).sort_values('t')
    print(f'{machine}: {len(sample)} random incidents over '
          f'{inc["t"].min():%Y-%m-%d}..{inc["t"].max():%Y-%m-%d}')
    for event in sample['t']:
        plot_incident_window(machine, event, inc, silver['telemetry'], half_hours=half_hours)

##### 5.5.1 MACH-03

In [ ]:
# 7-day window: +/-3.5 days = +/-84h.
show_incident_windows('MACH-03', 10, half_hours=84)

##### 5.5.2 MACH-13

In [ ]:
show_incident_windows('MACH-13', 10, half_hours=84)

## Batch & traceability

End-to-end **lineage** of the latest pipeline batch (`meta.processing_runs`): DataLake → bronze → silver → gold, with row counts (read / ingested / rejected), status, duration, quality flag, code version and output hash. Populated by `scripts/run_pipeline.py` (each execution = one `batch_id`).

In [ ]:
from src.framework.db.engine import get_engine, is_available
from src.framework.lineage.tracker import lineage_markdown
display(Markdown(lineage_markdown(get_engine()) if is_available() else '_PostgreSQL unavailable._'))

---
Reminder: this notebook is for exploration. The reproducible pipeline is `scripts/run_pipeline.py` (re-run after any data update in `data/raw/`).